# Pairwise judging with Kaggle models (Task 7)

1. Create a Kaggle dataset from the `kaggle_upload/` folder (pair files + `judge_pairwise_kaggle.py`) and attach it here.
2. Make sure this notebook has Kaggle model access for `kaggle_benchmarks` (benchmark / utility notebook).
3. Run all. Judgments are appended after every batch under `/kaggle/working/judged/`; re-running resumes and skips finished items.
4. When done: **Save Version**, then download `/kaggle/working/judged.zip`.

In [ ]:
import glob, json, os, subprocess, sys
JUDGE_MODEL = "google/gemini-3.6-flash"
OUT_DIR = "/kaggle/working/judged"
os.makedirs(OUT_DIR, exist_ok=True)
JUDGE = sorted(glob.glob("/kaggle/input/**/judge_pairwise_kaggle.py", recursive=True))[0]
PAIRS = sorted(glob.glob("/kaggle/input/**/*_pairs.jsonl", recursive=True))
print(JUDGE); print(*PAIRS, sep="\n")

In [ ]:
import kaggle_benchmarks as kbench
available = sorted(kbench.llms.keys())
print([m for m in available if "gemini" in m])
assert JUDGE_MODEL in available, f"{JUDGE_MODEL} unavailable - pick another judge above and record it in the report"

In [ ]:
# Required comparisons first, then persona sweep, then extra analysis.
ORDER = ["dpo_uf_vs_sft_alpaca", "dpo_hh_lr3e-6_vs_sft_hh", "dpo_hh_vs_sft_hh",
         "lora_dpo_vs_", "lora_dpo_lr1e-4", "lora_dpo_lr2e-4",
         "dpo_hh_lr3e-6_vs_sft_alpaca", "dpo_hh_lr1e-5_vs_sft_hh", "dpo_uf_vs_sft_hh",
         "dpo_hh_vs_sft_alpaca", "dpo_hh_lr1e-5_vs_sft_alpaca", "sft_vs_qwen"]
def rank(p):
    name = os.path.basename(p)
    return next((i for i, key in enumerate(ORDER) if name.startswith(key)), 99)
for pairs in sorted(PAIRS, key=rank):
    name = os.path.basename(pairs).replace("_pairs.jsonl", "")
    cmd = [sys.executable, JUDGE, "--pairs_path", pairs, "--output_path", f"{OUT_DIR}/{name}_judged.jsonl",
           "--judge_model", JUDGE_MODEL, "--batch_size", "5", "--sleep_seconds", "5",
           "--max_retries", "3", "--reuse_judgments", "--swap"]
    print("\n>>>", name, flush=True)
    subprocess.run(cmd, check=False)
    subprocess.run(["bash", "-c", "cd /kaggle/working && zip -qr judged.zip judged"], check=False)  # checkpoint zip

In [ ]:
for path in sorted(glob.glob(f"{OUT_DIR}/*_judged_summary.json")):
    s = json.load(open(path))
    for crit, c in s["criteria"].items():
        print(f"{s['comparison']:<50} {crit:<15} n={c['n']} W/T/L={c['wins']}/{c['ties']}/{c['losses']} "
              f"unk={c['unknown']} fail={c['failures']} ties_half={c['win_rate_ties_half']} "
              f"pos_consistency={c['position_consistency']}")